In [17]:
# pip install lazypredict

### Código para Classificação (LazyClassifier) com suporte ROC AUC em multiclasses

A ROC AUC (Área Sob a Curva ROC) é uma métrica usada para avaliar modelos de classificação, particularmente em problemas binários. É baseada na Curva ROC (Receiver Operating Characteristic), que é um gráfico que mostra a relação entre:

Taxa de verdadeiros positivos (True Positive Rate - TPR): Proporção de instâncias positivas corretamente classificadas.

Taxa de falsos positivos (False Positive Rate - FPR): Proporção de instâncias negativas incorretamente classificadas como positivas.

O valor AUC (Área Under the Curve) mede a área sob essa curva. Ele varia de 0 a 1:

1.0: O modelo é perfeito.

0.5: O modelo não tem capacidade discriminativa (equivalente a um chute aleatório).

< 0.5: O modelo é pior que um chute aleatório.

In [18]:
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from lazypredict.Supervised import LazyClassifier
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

# Carregar os dados do arquivo
file_path = '../../sensor_data GY-87_1.txt'
df = pd.read_csv(file_path, header=None, names=['x', 'y', 'z'])

# Usar apenas os primeiros 50 mil dados
df = df.iloc[:50000]

# Extrair apenas o eixo X
x = df['x']

# Criar labels para classificação
df['x_class'] = pd.qcut(x, q=5, labels=[0, 1, 2, 3, 4])

# Definir as features (X) e o target (y)
X = df[['x']]  # Feature: eixo X
y = df['x_class']  # Target: classes baseadas na magnitude de X

split_number = 1

# Configurar o TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)  # Divisão em 5 partes
for train_index, test_index in tscv.split(X):
    print(f"\n--- Processando Split {split_number} ---")

    # Separar os conjuntos de treino e teste em cada split
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Inicializar o LazyClassifier
    clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)

    # Ajustar o classificador e obter os melhores modelos
    models, predictions = clf.fit(X_train, X_test, y_train, y_test)

    # Exibir os resultados do LazyClassifier para o split atual
    print("Modelos avaliados com LazyClassifier (TimeSeriesSplit):")
    print(models)

    # Selecionar o melhor modelo identificado (exemplo: LGBMClassifier)
    best_model = LGBMClassifier(objective='multiclass', num_class=5, random_state=123, verbosity=-1)
    best_model.fit(X_train, y_train)

    # Obter as probabilidades previstas para o conjunto de teste
    y_pred_proba = best_model.predict_proba(X_test)

    # Binarizar os rótulos reais (necessário para o cálculo do ROC AUC)
    y_test_binarized = label_binarize(y_test, classes=[0, 1, 2, 3, 4])

    # Calcular o ROC AUC usando as médias micro e macro
    roc_auc_micro = roc_auc_score(y_test_binarized, y_pred_proba, average='micro', multi_class='ovr')
    roc_auc_macro = roc_auc_score(y_test_binarized, y_pred_proba, average='macro', multi_class='ovr')

    # Exibir os resultados do ROC AUC para o split atual
    print(f"\nROC AUC (Micro) para o split atual: {roc_auc_micro}")
    print(f"ROC AUC (Macro) para o split atual: {roc_auc_macro}\n")

    # Incrementar o número do split
    split_number += 1



--- Processando Split 1 ---


100%|██████████| 31/31 [00:16<00:00,  1.86it/s]


Modelos avaliados com LazyClassifier (TimeSeriesSplit):
                               Accuracy  Balanced Accuracy ROC AUC  F1 Score  \
Model                                                                          
LGBMClassifier                     1.00               1.00    None      1.00   
QuadraticDiscriminantAnalysis      1.00               1.00    None      1.00   
LogisticRegression                 1.00               1.00    None      1.00   
BaggingClassifier                  1.00               1.00    None      1.00   
LabelSpreading                     1.00               1.00    None      1.00   
LabelPropagation                   1.00               1.00    None      1.00   
KNeighborsClassifier               1.00               1.00    None      1.00   
GaussianNB                         1.00               1.00    None      1.00   
ExtraTreesClassifier               1.00               1.00    None      1.00   
ExtraTreeClassifier                1.00               1.00    No

100%|██████████| 31/31 [00:50<00:00,  1.63s/it]


Modelos avaliados com LazyClassifier (TimeSeriesSplit):
                               Accuracy  Balanced Accuracy ROC AUC  F1 Score  \
Model                                                                          
LGBMClassifier                     1.00               1.00    None      1.00   
QuadraticDiscriminantAnalysis      1.00               1.00    None      1.00   
LogisticRegression                 1.00               1.00    None      1.00   
BaggingClassifier                  1.00               1.00    None      1.00   
LabelSpreading                     1.00               1.00    None      1.00   
LabelPropagation                   1.00               1.00    None      1.00   
KNeighborsClassifier               1.00               1.00    None      1.00   
GaussianNB                         1.00               1.00    None      1.00   
ExtraTreesClassifier               1.00               1.00    None      1.00   
ExtraTreeClassifier                1.00               1.00    No

...

Modelos avaliados com LazyClassifier (TimeSeriesSplit):
                               Accuracy  Balanced Accuracy ROC AUC  F1 Score  \
Model                                                                          
LGBMClassifier                     1.00               1.00    None      1.00   
QuadraticDiscriminantAnalysis      1.00               1.00    None      1.00   
LogisticRegression                 1.00               1.00    None      1.00   
BaggingClassifier                  1.00               1.00    None      1.00   
LabelSpreading                     1.00               1.00    None      1.00   
LabelPropagation                   1.00               1.00    None      1.00   
KNeighborsClassifier               1.00               1.00    None      1.00   
GaussianNB                         1.00               1.00    None      1.00   
ExtraTreesClassifier               1.00               1.00    None      1.00   
ExtraTreeClassifier                1.00               1.00    No

100%|██████████| 31/31 [03:00<00:00,  5.82s/it]


Modelos avaliados com LazyClassifier (TimeSeriesSplit):
                               Accuracy  Balanced Accuracy ROC AUC  F1 Score  \
Model                                                                          
LGBMClassifier                     1.00               1.00    None      1.00   
QuadraticDiscriminantAnalysis      1.00               1.00    None      1.00   
LogisticRegression                 1.00               1.00    None      1.00   
BaggingClassifier                  1.00               1.00    None      1.00   
LabelSpreading                     1.00               1.00    None      1.00   
LabelPropagation                   1.00               1.00    None      1.00   
KNeighborsClassifier               1.00               1.00    None      1.00   
GaussianNB                         1.00               1.00    None      1.00   
ExtraTreesClassifier               1.00               1.00    None      1.00   
ExtraTreeClassifier                1.00               1.00    No

100%|██████████| 31/31 [03:52<00:00,  7.51s/it]


Modelos avaliados com LazyClassifier (TimeSeriesSplit):
                               Accuracy  Balanced Accuracy ROC AUC  F1 Score  \
Model                                                                          
LGBMClassifier                     1.00               1.00    None      1.00   
QuadraticDiscriminantAnalysis      1.00               1.00    None      1.00   
LogisticRegression                 1.00               1.00    None      1.00   
BaggingClassifier                  1.00               1.00    None      1.00   
LabelSpreading                     1.00               1.00    None      1.00   
LabelPropagation                   1.00               1.00    None      1.00   
KNeighborsClassifier               1.00               1.00    None      1.00   
GaussianNB                         1.00               1.00    None      1.00   
ExtraTreesClassifier               1.00               1.00    None      1.00   
ExtraTreeClassifier                1.00               1.00    No

### Código para Regressão (LazyRegressor)

In [19]:
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from lazypredict.Supervised import LazyRegressor
from sklearn.metrics import mean_absolute_error

# Carregar os dados do arquivo
file_path = '../../sensor_data GY-87_1.txt'
df = pd.read_csv(file_path, header=None, names=['x', 'y', 'z'])

# Usar apenas os primeiros 50 mil dados
df = df.iloc[:50000]

# Extrair apenas o eixo Z
z = df['z']

# Definir as features (X) e o target (y) para regressão
X = df[['z']]  # Usando apenas o eixo Z como feature
y = z  # O target é o próprio eixo Z, como regressão contínua

# Inicializar o TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

# Lista para armazenar os resultados
all_models = []

split_number = 1

for train_index, test_index in tscv.split(X):
    print(f"\n--- Processando Split {split_number} ---")

    # Dividir os dados em treino e teste respeitando a ordem temporal
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Inicializar o LazyRegressor
    reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
    
    # Ajustar o regressor e obter os melhores modelos
    models, predictions = reg.fit(X_train, X_test, y_train, y_test)
    
    # Adicionar os modelos desta divisão à lista
    all_models.append(models)
    
    # Exibir os resultados desta divisão
    print(f"Resultados da divisão atual:")
    print(models)

# Exemplo de análise dos resultados
print("\nResultados combinados de todas as divisões do TimeSeriesSplit:")
for i, models in enumerate(all_models):
    print(f"\nDivisão {i+1}:")
    print(models)

# Incrementar o número do split
split_number += 1



--- Processando Split 1 ---


100%|██████████| 42/42 [00:26<00:00,  1.61it/s]


Resultados da divisão atual:
                               Adjusted R-Squared  R-Squared  RMSE  Time Taken
Model                                                                         
OrthogonalMatchingPursuit                    1.00       1.00  0.00        0.01
LinearRegression                             1.00       1.00  0.00        0.01
LassoLarsCV                                  1.00       1.00  0.00        0.01
LinearSVR                                    1.00       1.00  0.00        0.01
LarsCV                                       1.00       1.00  0.00        0.01
Lars                                         1.00       1.00  0.00        0.01
RANSACRegressor                              1.00       1.00  0.00        0.01
HuberRegressor                               1.00       1.00  0.00        0.05
LassoLarsIC                                  1.00       1.00  0.00        0.01
TransformedTargetRegressor                   1.00       1.00  0.00        0.01
BayesianRidge          

100%|██████████| 42/42 [02:26<00:00,  3.49s/it]


Resultados da divisão atual:
                               Adjusted R-Squared  R-Squared  RMSE  Time Taken
Model                                                                         
LarsCV                                       1.00       1.00  0.00        0.02
RANSACRegressor                              1.00       1.00  0.00        0.01
BayesianRidge                                1.00       1.00  0.00        0.01
Lars                                         1.00       1.00  0.00        0.01
OrthogonalMatchingPursuit                    1.00       1.00  0.00        0.01
TransformedTargetRegressor                   1.00       1.00  0.00        0.01
LinearSVR                                    1.00       1.00  0.00        0.01
LinearRegression                             1.00       1.00  0.00        0.01
LassoLarsIC                                  1.00       1.00  0.00        0.01
LassoLarsCV                                  1.00       1.00  0.00        0.02
HuberRegressor         

...

Resultados da divisão atual:
                               Adjusted R-Squared  R-Squared  RMSE  Time Taken
Model                                                                         
OrthogonalMatchingPursuit                    1.00       1.00  0.00        0.01
LassoLarsIC                                  1.00       1.00  0.00        0.01
LinearRegression                             1.00       1.00  0.00        0.01
LarsCV                                       1.00       1.00  0.00        0.02
Lars                                         1.00       1.00  0.00        0.01
RANSACRegressor                              1.00       1.00  0.00        0.02
LinearSVR                                    1.00       1.00  0.00        0.01
HuberRegressor                               1.00       1.00  0.00        0.12
LassoLarsCV                                  1.00       1.00  0.00        0.02
TransformedTargetRegressor                   1.00       1.00  0.00        0.01
BayesianRidge          

100%|██████████| 42/42 [10:00<00:00, 14.30s/it]


Resultados da divisão atual:
                               Adjusted R-Squared  R-Squared  RMSE  Time Taken
Model                                                                         
Lars                                         1.00       1.00  0.00        0.02
HuberRegressor                               1.00       1.00  0.00        0.17
OrthogonalMatchingPursuit                    1.00       1.00  0.00        0.01
LinearSVR                                    1.00       1.00  0.00        0.01
LinearRegression                             1.00       1.00  0.00        0.01
LassoLarsIC                                  1.00       1.00  0.00        0.02
LassoLarsCV                                  1.00       1.00  0.00        0.02
LarsCV                                       1.00       1.00  0.00        0.02
RANSACRegressor                              1.00       1.00  0.00        0.02
BayesianRidge                                1.00       1.00  0.00        0.01
TransformedTargetRegres

100%|██████████| 42/42 [22:17<00:00, 31.84s/it]   

Resultados da divisão atual:
                               Adjusted R-Squared  R-Squared  RMSE  Time Taken
Model                                                                         
Lars                                         1.00       1.00  0.00        0.02
HuberRegressor                               1.00       1.00  0.00        0.24
OrthogonalMatchingPursuit                    1.00       1.00  0.00        0.01
LinearSVR                                    1.00       1.00  0.00        0.01
LinearRegression                             1.00       1.00  0.00        0.01
LassoLarsIC                                  1.00       1.00  0.00        0.02
LassoLarsCV                                  1.00       1.00  0.00        0.02
LarsCV                                       1.00       1.00  0.00        0.02
RANSACRegressor                              1.00       1.00  0.00        0.02
TransformedTargetRegressor                   1.00       1.00  0.00        0.01
ExtraTreesRegressor    